# Phase 5 - Notebook 09: VGGT to 3DGS Pipeline Integration\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase5/09_vggt_to_3dgs.ipynb)\n\n---\n\n## Learning Objectives\n\nBy the end of this notebook, you will:\n1. Understand different approaches to integrate VGGT with 3D Gaussian Splatting (3DGS)\n2. Learn how to convert VGGT outputs to COLMAP format for existing pipelines\n3. Know how to initialize 3D Gaussians directly from VGGT predictions\n4. Master depth-based point cloud generation from depth maps\n5. Understand camera parameter extraction and conversion\n6. Learn confidence filtering strategies for high-quality point clouds\n7. Have a complete end-to-end pipeline from images to 3DGS\n8. Be able to integrate VGGT outputs with gsplat\n\n**Estimated Time**: 60 minutes\n\n**Prerequisites**: Phase 3 (DUSt3R), Phase 5 (VGGT architecture), basic understanding of 3DGS\n\n---

## 0. Environment Setup

In [1]:
# Environment setup\nimport os\nimport sys\n\n# Colab compatibility\nif 'COLAB_GPU' in os.environ:\n    !pip install -q torch torchvision numpy matplotlib plotly ipywidgets\n    !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git\n    %cd 3DGS-from-scratch\n\n# Add project root to path\nproject_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))\nif project_root not in sys.path:\n    sys.path.insert(0, project_root)\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport matplotlib.patches as mpatches\nfrom matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle, Polygon\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Try to import torch\ntry:\n    import torch\n    import torch.nn as nn\n    TORCH_AVAILABLE = True\n    print(f"PyTorch version: {torch.__version__}")\nexcept ImportError:\n    TORCH_AVAILABLE = False\n    print("PyTorch not available - running in simulation mode")\n\nprint("Environment ready!")\nprint(f"NumPy version: {np.__version__}")

## 1. Integration Overview: From VGGT to 3DGS\n\nVGGT provides rich 3D information from images, but we need to bridge the gap to 3D Gaussian Splatting. Let's explore the integration pipeline.\n\n### Two Main Integration Approaches\n\n```\nApproach 1: COLMAP Export (Compatible)\n========================================\nVGGT Predictions → COLMAP format → COLMAP-based 3DGS\n     ↓                    ↓\n  Camera Poses      cameras.txt\n  Depth Maps        images.txt\n  Point Cloud       points3D.txt\n\nApproach 2: Direct Initialization (Efficient)\n===============================================\nVGGT Predictions → Direct 3D Gaussian Params → gsplat\n     ↓                        ↓\n  Camera Poses          xyz (positions)\n  Depth → Points        rgb (colors)\n  Confidence            opacity\n                        scales\n                        quaternions\n```\n\n### Comparison\n\n| Aspect | COLMAP Export | Direct Initialization |\n|--------|---------------|---------------------|\n| **Compatibility** | Works with existing pipelines | Requires custom code |\n| **Speed** | Slower (file I/O + conversion) | Faster (in-memory) |\n| **Flexibility** | Limited by COLMAP format | Full control over params |\n| **Use Case** | Integration with existing tools | Research/Custom workflows |\n| **Point Quality** | Depends on COLMAP processing | Direct from VGGT |\n| **Training Time** | Standard | Potentially faster init |\n\n### Recommended Workflow\n\n1. **Prototyping**: Use COLMAP export to leverage existing tools\n2. **Production**: Use direct initialization for efficiency\n3. **Hybrid**: Export keyframes to COLMAP, initialize Gaussians directly

In [2]:
# 集成管道可视化 - Integration Pipeline Visualization\n\nfig, axes = plt.subplots(1, 2, figsize=(18, 10))\n\n# 颜色定义\ncolor_input = '#E3F2FD'\ncolor_vggt = '#FFF3E0'\ncolor_output = '#E8F5E9'\ncolor_colmap = '#FCE4EC'\ncolor_3dgs = '#F3E5F5'\narrow_color = '#424242'\n\n# === Left: COLMAP Export Pipeline ===\nax = axes[0]\nax.set_xlim(0, 10)\nax.set_ylim(0, 12)\nax.set_aspect('equal')\nax.axis('off')\nax.set_title('Approach 1: COLMAP Export Pipeline', fontsize=14, fontweight='bold', pad=20)\n\n# Input images\nfor i in range(3):\n    rect = FancyBboxPatch((0.5, 9-i*1.2), 1.5, 0.8, boxstyle="round,pad=0.05",\n                         facecolor=color_input, edgecolor='#1976D2', linewidth=2)\n    ax.add_patch(rect)\n    ax.text(1.25, 9.4-i*1.2, f'Img {i+1}', ha='center', va='center', fontsize=9, fontweight='bold')\n\n# Arrow to VGGT\nax.annotate('', xy=(3, 8.5), xytext=(2.2, 8.5),\n            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2))\nax.text(2.6, 8.9, 'VGGT', fontsize=10, ha='center', style='italic')\n\n# VGGT block\nvggt_box = FancyBboxPatch((3, 6.5), 2.5, 4, boxstyle="round,pad=0.1",\n                         facecolor=color_vggt, edgecolor='#E65100', linewidth=3)\nax.add_patch(vggt_box)\nax.text(4.25, 9.8, 'VGGT Model', ha='center', fontsize=11, fontweight='bold')\nax.text(4.25, 9.3, '• Cameras', ha='left', fontsize=9)\nax.text(4.25, 8.8, '• Depth Maps', ha='left', fontsize=9)\nax.text(4.25, 8.3, '• Point Cloud', ha='left', fontsize=9)\nax.text(4.25, 7.8, '• Confidence', ha='left', fontsize=9)\n\n# Arrow to COLMAP\nax.annotate('', xy=(6.3, 8.5), xytext=(5.7, 8.5),\n            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2))\n\n# COLMAP format\ncolmap_box = FancyBboxPatch((6.5, 6.5), 3, 4, boxstyle="round,pad=0.1",\n                           facecolor=color_colmap, edgecolor='#C2185B', linewidth=3)\nax.add_patch(colmap_box)\nax.text(8, 9.8, 'COLMAP Format', ha='center', fontsize=11, fontweight='bold')\nax.text(8, 9.2, 'cameras.txt', ha='center', fontsize=9, family='monospace')\nax.text(8, 8.6, 'images.txt', ha='center', fontsize=9, family='monospace')\nax.text(8, 8.0, 'points3D.txt', ha='center', fontsize=9, family='monospace')\nax.text(8, 7.0, 'Sparse Reconstruction', ha='center', fontsize=8, style='italic', color='#666')\n\n# Arrow to 3DGS\nax.annotate('', xy=(4.25, 5.8), xytext=(4.25, 6.4),\n            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2))\nax.text(5.5, 6.1, 'Load', fontsize=9, style='italic')\n\n# 3DGS\ngs_box = FancyBboxPatch((2.5, 3.5), 3.5, 2, boxstyle="round,pad=0.1",\n                       facecolor=color_3dgs, edgecolor='#7B1FA2', linewidth=3)\nax.add_patch(gs_box)\nax.text(4.25, 4.8, '3D Gaussian Splatting', ha='center', fontsize=11, fontweight='bold')\nax.text(4.25, 4.2, 'Standard Pipeline', ha='center', fontsize=9)\n\n# Characteristics\nax.text(5, 2.5, '✓ Compatible with existing tools', fontsize=9, color='#2E7D32')\nax.text(5, 2.0, '✓ Uses COLMAP loaders', fontsize=9, color='#2E7D32')\nax.text(5, 1.5, '✗ File I/O overhead', fontsize=9, color='#C62828')\nax.text(5, 1.0, '✗ Format conversion needed', fontsize=9, color='#C62828')\n\n# === Right: Direct Initialization Pipeline ===\nax = axes[1]\nax.set_xlim(0, 10)\nax.set_ylim(0, 12)\nax.set_aspect('equal')\nax.axis('off')\nax.set_title('Approach 2: Direct Initialization Pipeline', fontsize=14, fontweight='bold', pad=20)\n\n# Input images\nfor i in range(3):\n    rect = FancyBboxPatch((0.5, 9-i*1.2), 1.5, 0.8, boxstyle="round,pad=0.05",\n                         facecolor=color_input, edgecolor='#1976D2', linewidth=2)\n    ax.add_patch(rect)\n    ax.text(1.25, 9.4-i*1.2, f'Img {i+1}', ha='center', va='center', fontsize=9, fontweight='bold')\n\n# Arrow to VGGT\nax.annotate('', xy=(3, 8.5), xytext=(2.2, 8.5),\n            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2))\nax.text(2.6, 8.9, 'VGGT', fontsize=10, ha='center', style='italic')\n\n# VGGT block (same)\nvggt_box = FancyBboxPatch((3, 6.5), 2.5, 4, boxstyle="round,pad=0.1",\n                         facecolor=color_vggt, edgecolor='#E65100', linewidth=3)\nax.add_patch(vggt_box)\nax.text(4.25, 9.8, 'VGGT Model', ha='center', fontsize=11, fontweight='bold')\nax.text(4.25, 9.3, '• Cameras', ha='left', fontsize=9)\nax.text(4.25, 8.8, '• Depth Maps', ha='left', fontsize=9)\nax.text(4.25, 8.3, '• Point Cloud', ha='left', fontsize=9)\nax.text(4.25, 7.8, '• Confidence', ha='left', fontsize=9)\n\n# Arrow to Direct Init\nax.annotate('', xy=(6.3, 8.5), xytext=(5.7, 8.5),\n            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2))\n\n# Direct Initialization\ninit_box = FancyBboxPatch((6.5, 6.5), 3, 4, boxstyle="round,pad=0.1",\n                         facecolor='#E0F7FA', edgecolor='#00838F', linewidth=3)\nax.add_patch(init_box)\nax.text(8, 9.8, 'Direct Initialization', ha='center', fontsize=11, fontweight='bold')\nax.text(8, 9.3, '• xyz (from depth)', ha='left', fontsize=9)\nax.text(8, 8.8, '• rgb (from images)', ha='left', fontsize=9)\nax.text(8, 8.3, '• scales (from conf)', ha='left', fontsize=9)\nax.text(8, 7.8, '• quats (from normals)', ha='left', fontsize=9)\nax.text(8, 7.3, '• opacity (from conf)', ha='left', fontsize=9)\n\n# Arrow to gsplat\nax.annotate('', xy=(4.25, 5.8), xytext=(4.25, 6.4),\n            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2))\nax.text(5.5, 6.1, 'Initialize', fontsize=9, style='italic')\n\n# gsplat\ngsplat_box = FancyBboxPatch((2.5, 3.5), 3.5, 2, boxstyle="round,pad=0.1",\n                           facecolor='#F1F8E9', edgecolor='#558B2F', linewidth=3)\nax.add_patch(gsplat_box)\nax.text(4.25, 4.8, 'gsplat / Custom 3DGS', ha='center', fontsize=11, fontweight='bold')\nax.text(4.25, 4.2, 'Direct Rendering', ha='center', fontsize=9)\n\n# Characteristics\nax.text(5, 2.5, '✓ No file I/O overhead', fontsize=9, color='#2E7D32')\nax.text(5, 2.0, '✓ Full parameter control', fontsize=9, color='#2E7D32')\nax.text(5, 1.5, '✓ In-memory processing', fontsize=9, color='#2E7D32')\nax.text(5, 1.0, '✗ Requires custom code', fontsize=9, color='#C62828')\n\nplt.tight_layout()\nplt.savefig('vggt_integration_approaches.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Two integration approaches for VGGT → 3DGS:")\nprint("\nApproach 1: COLMAP Export")\nprint("  • Convert VGGT outputs to COLMAP format")\nprint("  • Use existing COLMAP loaders")\nprint("  • Compatible with existing pipelines")\nprint("\nApproach 2: Direct Initialization")\nprint("  • Convert VGGT outputs directly to Gaussian parameters")\nprint("  • Initialize gsplat/custom 3DGS directly")\nprint("  • More efficient, more control")

## 2. COLMAP Export Pipeline\n\nCOLMAP uses three main files for sparse reconstruction:\n- **cameras.txt**: Camera intrinsics (focal length, principal point, distortion)\n- **images.txt**: Camera extrinsics (rotation, translation) and 2D keypoints\n- **points3D.txt**: 3D points with color and track information\n\n### File Format Reference\n\n**cameras.txt**: `CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]`\n```\n# Camera list with one line of data per camera:\n#   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n# Number of cameras: 1\n1 PINHOLE 1920 1080 1863.21 1863.21 960.0 540.0\n```\n\n**images.txt**: `IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME`\n```\n# Image list with two lines of data per image:\n#   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n#   POINTS2D[] as (X, Y, POINT3D_ID)\n1 0.9999 0.001 0.002 0.003 0.1 0.2 0.3 1 image001.jpg\n0.0 0.0 -1  # (no 3D points for this example)\n```\n\n**points3D.txt**: `POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[]`\n```\n# 3D point list with one line of data per point:\n#   POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[] as (IMAGE_ID, POINT2D_IDX)\n1 1.0 2.0 3.0 255 128 64 0.5 1 0 2 1\n```

In [3]:
# COLMAP导出实现 - COLMAP Export Implementation\n\nclass COLMAPExporter:\n    """将VGGT输出导出为COLMAP格式\n    Export VGGT outputs to COLMAP format\n    """\n    \n    def __init__(self, output_dir="sparse/0"):\n        self.output_dir = output_dir\n    \n    def export_cameras(self, intrinsics, image_size, camera_model="PINHOLE"):\n        """\n        导出相机内参\n        Export camera intrinsics\n        \n        Args:\n            intrinsics: dict with 'fx', 'fy', 'cx', 'cy' or [N, 4] tensor\n            image_size: (width, height) tuple\n            camera_model: COLMAP camera model type\n        """\n        width, height = image_size\n        \n        # Handle single camera or multiple\n        if isinstance(intrinsics, dict):\n            fx, fy = intrinsics['fx'], intrinsics['fy']\n            cx, cy = intrinsics['cx'], intrinsics['cy']\n            num_cameras = 1\n        else:\n            # Assume numpy array or tensor [N, 4]\n            fx, fy = intrinsics[0, 0], intrinsics[0, 1]\n            cx, cy = intrinsics[0, 2], intrinsics[0, 3]\n            num_cameras = len(intrinsics)\n        \n        lines = ["# Camera list with one line of data per camera:",\n                 "#   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]",\n                 f"# Number of cameras: {num_cameras}"]\n        \n        for i in range(num_cameras):\n            if isinstance(intrinsics, dict):\n                fx_i, fy_i, cx_i, cy_i = fx, fy, cx, cy\n            else:\n                fx_i, fy_i, cx_i, cy_i = intrinsics[i]\n            lines.append(f"{i+1} {camera_model} {width} {height} {fx_i:.6f} {fy_i:.6f} {cx_i:.6f} {cy_i:.6f}")\n        \n        return '\n'.join(lines)\n    \n    def export_images(self, poses, image_names, camera_ids=None):\n        """\n        导出图像位姿\n        Export image poses\n        \n        Args:\n            poses: [N, 4, 4] transformation matrices or [N, 7] (quat + trans)\n            image_names: list of image filenames\n            camera_ids: camera ID for each image (default: 1 for all)\n        """\n        num_images = len(image_names)\n        if camera_ids is None:\n            camera_ids = [1] * num_images\n        \n        lines = ["# Image list with two lines of data per image:",\n                 "#   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME",\n                 "#   POINTS2D[] as (X, Y, POINT3D_ID)",\n                 f"# Number of images: {num_images}"]\n        \n        for i, (pose, name, cam_id) in enumerate(zip(poses, image_names, camera_ids)):\n            if pose.shape == (4, 4):\n                # Convert 4x4 matrix to quaternion + translation\n                R = pose[:3, :3]\n                t = pose[:3, 3]\n                q = self.rotation_matrix_to_quaternion(R)\n            elif pose.shape == (7,):\n                q = pose[:4]\n                t = pose[4:]\n            else:\n                raise ValueError(f"Unexpected pose shape: {pose.shape}")\n            \n            # COLMAP format: q = (w, x, y, z), t = (tx, ty, tz)\n            qw, qx, qy, qz = q\n            tx, ty, tz = t\n            \n            lines.append(f"{i+1} {qw:.10f} {qx:.10f} {qy:.10f} {qz:.10f} {tx:.10f} {ty:.10f} {tz:.10f} {cam_id} {name}")\n            lines.append("")  # Empty line for POINTS2D (optional)\n        \n        return '\n'.join(lines)\n    \n    def export_points3D(self, points, colors=None, errors=None):\n        """\n        导出3D点云\n        Export 3D point cloud\n        \n        Args:\n            points: [N, 3] 3D point coordinates\n            colors: [N, 3] RGB colors (0-255)\n            errors: [N] reprojection errors\n        """\n        num_points = len(points)\n        \n        if colors is None:\n            colors = np.full((num_points, 3), 128, dtype=np.uint8)\n        if errors is None:\n            errors = np.zeros(num_points)\n        \n        lines = ["# 3D point list with one line of data per point:",\n                 "#   POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[]",\n                 f"# Number of points: {num_points}"]\n        \n        for i, (pt, col, err) in enumerate(zip(points, colors, errors)):\n            x, y, z = pt\n            r, g, b = col.astype(int)\n            # No track information (empty track)\n            lines.append(f"{i+1} {x:.6f} {y:.6f} {z:.6f} {r} {g} {b} {err:.6f}")\n        \n        return '\n'.join(lines)\n    \n    @staticmethod\n    def rotation_matrix_to_quaternion(R):\n        """Convert rotation matrix to quaternion (w, x, y, z)\n        将旋转矩阵转换为四元数\n        """\n        trace = np.trace(R)\n        if trace > 0:\n            s = 0.5 / np.sqrt(trace + 1.0)\n            w = 0.25 / s\n            x = (R[2, 1] - R[1, 2]) * s\n            y = (R[0, 2] - R[2, 0]) * s\n            z = (R[1, 0] - R[0, 1]) * s\n        elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:\n            s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])\n            w = (R[2, 1] - R[1, 2]) / s\n            x = 0.25 * s\n            y = (R[0, 1] + R[1, 0]) / s\n            z = (R[0, 2] + R[2, 0]) / s\n        elif R[1, 1] > R[2, 2]:\n            s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])\n            w = (R[0, 2] - R[2, 0]) / s\n            x = (R[0, 1] + R[1, 0]) / s\n            y = 0.25 * s\n            z = (R[1, 2] + R[2, 1]) / s\n        else:\n            s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])\n            w = (R[1, 0] - R[0, 1]) / s\n            x = (R[0, 2] + R[2, 0]) / s\n            y = (R[1, 2] + R[2, 1]) / s\n            z = 0.25 * s\n        return np.array([w, x, y, z])\n    \n    def write_to_disk(self, cameras_txt, images_txt, points3D_txt):\n        """Write COLMAP files to disk\n        写入COLMAP文件到磁盘\n        """\n        import os\n        os.makedirs(self.output_dir, exist_ok=True)\n        \n        with open(os.path.join(self.output_dir, 'cameras.txt'), 'w') as f:\n            f.write(cameras_txt)\n        with open(os.path.join(self.output_dir, 'images.txt'), 'w') as f:\n            f.write(images_txt)\n        with open(os.path.join(self.output_dir, 'points3D.txt'), 'w') as f:\n            f.write(points3D_txt)\n        \n        print(f"COLMAP files written to {self.output_dir}/")\n\n\n# 演示COLMAP导出 - Demonstrate COLMAP Export\nprint("=" * 60)\nprint("COLMAP导出演示")\nprint("=" * 60)\n\nexporter = COLMAPExporter(output_dir="demo_sparse")\n\n# 模拟VGGT输出 - Simulate VGGT outputs\nnum_images = 5\nimage_size = (1920, 1080)\nintrinsics = {'fx': 1863.21, 'fy': 1863.21, 'cx': 960.0, 'cy': 540.0}\n\n# 生成模拟位姿 - Generate dummy poses\nposes = []\nfor i in range(num_images):\n    angle = i * 2 * np.pi / num_images\n    R = np.array([[np.cos(angle), 0, np.sin(angle)],\n                  [0, 1, 0],\n                  [-np.sin(angle), 0, np.cos(angle)]])\n    t = np.array([3*np.cos(angle), 0, 3*np.sin(angle)])\n    pose = np.eye(4)\n    pose[:3, :3] = R\n    pose[:3, 3] = t\n    poses.append(pose)\n\nimage_names = [f"image_{i:03d}.jpg" for i in range(num_images)]\n\n# 生成模拟点云 - Generate dummy point cloud\nnum_points = 100\npoints = np.random.randn(num_points, 3) * 2\ncolors = np.random.randint(0, 256, (num_points, 3))\n\n# 导出 - Export\ncameras_txt = exporter.export_cameras(intrinsics, image_size)\nimages_txt = exporter.export_images(poses, image_names)\npoints3D_txt = exporter.export_points3D(points, colors)\n\nprint("\ncameras.txt (first 3 lines):")\nprint('\n'.join(cameras_txt.split('\n')[:3]))\nprint("...")\nprint("\nimages.txt (first 4 lines):")\nprint('\n'.join(images_txt.split('\n')[:4]))\nprint("...")\nprint(f"\npoints3D.txt: {num_points} points")\n\n# 保存到文件 - Save to files (optional)\n# exporter.write_to_disk(cameras_txt, images_txt, points3D_txt)\n\nprint("\nCOLMAP导出完成!")\nprint("你可以将这些文件加载到COLMAP或3DGS训练脚本中")

## 3. Direct 3DGS Initialization\n\nInstead of going through COLMAP format, we can directly initialize 3D Gaussian parameters from VGGT outputs.\n\n### 3D Gaussian Parameters\n\nEach 3D Gaussian is defined by:\n- **xyz**: 3D position (mean)\n- **rgb** or **features**: Color (or spherical harmonics coefficients)\n- **opacity**: Alpha value (before sigmoid)\n- **scaling**: 3D scales (before exp)\n- **rotation**: Quaternion (w, x, y, z)\n\n### Mapping VGGT Outputs to Gaussian Parameters\n\n| VGGT Output | Gaussian Parameter | Method |\n|-------------|-------------------|--------|\n| Depth maps | xyz | Unproject using camera intrinsics |\n| RGB images | rgb | Direct color sampling |\n| Confidence | opacity | Scale confidence to [0, 1] |\n| Depth variance | scaling | High variance → larger scale |\n| Surface normals | rotation | Align to normal direction |\n\n### Initialization Strategy\n\n```\nFor each pixel (u, v) in each image:\n  1. Sample depth d at (u, v)\n  2. Unproject (u, v, d) → (x, y, z) using camera intrinsics\n  3. Sample RGB color from image at (u, v)\n  4. Get confidence c at (u, v)\n  5. Initialize:\n     - xyz = (x, y, z)\n     - rgb = sampled color\n     - opacity = sigmoid⁻¹(c)  # inverse sigmoid\n     - scaling = base_scale * (1 + (1-c))  # lower confidence → larger\n     - rotation = quaternion from normal or identity\n```

In [4]:
# 直接3DGS初始化 - Direct 3DGS Initialization\n\nclass GaussianInitializer:\n    """\n    从VGGT输出直接初始化3D高斯\n    Initialize 3D Gaussians directly from VGGT outputs\n    """\n    \n    def __init__(self, subsample=8, confidence_threshold=0.5):\n        """\n        Args:\n            subsample: 像素采样间隔 (pixel subsampling stride)\n            confidence_threshold: 置信度阈值 (confidence filter threshold)\n        """\n        self.subsample = subsample\n        self.confidence_threshold = confidence_threshold\n    \n    def initialize_from_vggt(self, depth_maps, colors, confidences,\n                            intrinsics, extrinsics, image_size):\n        """\n        从VGGT输出初始化高斯\n        Initialize Gaussians from VGGT outputs\n        \n        Args:\n            depth_maps: [N, H, W] depth values\n            colors: [N, 3, H, W] RGB images (0-1)\n            confidences: [N, H, W] confidence values (0-1)\n            intrinsics: [N, 3, 3] camera intrinsics\n            extrinsics: [N, 4, 4] camera extrinsics (world-to-cam)\n            image_size: (H, W) tuple\n        \n        Returns:\n            gaussians: dict with 'xyz', 'rgb', 'opacity', 'scaling', 'rotation'\n        """\n        H, W = image_size\n        N = len(depth_maps)\n        \n        all_xyz = []\n        all_rgb = []\n        all_opacity = []\n        all_scaling = []\n        all_rotation = []\n        \n        # 像素网格\n        # Create pixel grid\n        v_coords = np.arange(0, H, self.subsample)\n        u_coords = np.arange(0, W, self.subsample)\n        vv, uu = np.meshgrid(v_coords, u_coords, indexing='ij')\n        \n        for i in range(N):\n            depth = depth_maps[i]\n            color = colors[i]\n            conf = confidences[i]\n            K = intrinsics[i]\n            E = extrinsics[i]\n            \n            # 采样子集 - Sample subset\n            d_sampled = depth[vv, uu]\n            c_sampled = conf[vv, uu]\n            \n            # 置信度过滤 - Filter by confidence\n            valid_mask = c_sampled > self.confidence_threshold\n            \n            if not valid_mask.any():\n                continue\n            \n            # 获取有效像素 - Get valid pixels\n            u_valid = uu[valid_mask]\n            v_valid = vv[valid_mask]\n            d_valid = d_sampled[valid_mask]\n            c_valid = c_sampled[valid_mask]\n            \n            # 反投影到3D - Unproject to 3D (camera space)\n            # X = (u - cx) * d / fx\n            # Y = (v - cy) * d / fy\n            # Z = d\n            fx, fy = K[0, 0], K[1, 1]\n            cx, cy = K[0, 2], K[1, 2]\n            \n            X_cam = (u_valid - cx) * d_valid / fx\n            Y_cam = (v_valid - cy) * d_valid / fy\n            Z_cam = d_valid\n            \n            # 转换到世界坐标 - Transform to world coordinates\n            points_cam = np.stack([X_cam, Y_cam, Z_cam, np.ones_like(X_cam)], axis=0)\n            points_world = (np.linalg.inv(E) @ points_cam).T  # [M, 4]\n            xyz = points_world[:, :3]\n            \n            # 采样颜色 - Sample colors\n            if color.ndim == 3:  # [3, H, W]\n                rgb = color[:, v_valid, u_valid].T  # [M, 3]\n            else:  # [H, W, 3]\n                rgb = color[v_valid, u_valid]\n            \n            # 初始化其他参数 - Initialize other parameters\n            # Opacity: inverse sigmoid of confidence\n            opacity = self.inverse_sigmoid(c_valid)\n            \n            # Scaling: base scale adjusted by confidence\n            base_scale = 0.01 * self.subsample  # Scale with subsample\n            scaling = base_scale * (1.0 + (1.0 - c_valid)[:, None].repeat(3, axis=1))\n            \n            # Rotation: identity quaternion (can be improved with normals)\n            rotation = np.tile([1, 0, 0, 0], (len(xyz), 1))  # [M, 4]\n            \n            all_xyz.append(xyz)\n            all_rgb.append(rgb)\n            all_opacity.append(opacity)\n            all_scaling.append(scaling)\n            all_rotation.append(rotation)\n        \n        # 合并所有帧 - Concatenate all frames\n        gaussians = {\n            'xyz': np.concatenate(all_xyz, axis=0) if all_xyz else np.array([]),\n            'rgb': np.concatenate(all_rgb, axis=0) if all_rgb else np.array([]),\n            'opacity': np.concatenate(all_opacity, axis=0) if all_opacity else np.array([]),\n            'scaling': np.concatenate(all_scaling, axis=0) if all_scaling else np.array([]),\n            'rotation': np.concatenate(all_rotation, axis=0) if all_rotation else np.array([])\n        }\n        \n        return gaussians\n    \n    @staticmethod\n    def inverse_sigmoid(x, eps=1e-6):\n        """\n        逆sigmoid函数\n        Inverse sigmoid function\n        """\n        x = np.clip(x, eps, 1 - eps)\n        return np.log(x / (1 - x))\n    \n    def get_statistics(self, gaussians):\n        """\n        获取初始化统计\n        Get initialization statistics\n        """\n        num_gaussians = len(gaussians['xyz'])\n        stats = {\n            'num_gaussians': num_gaussians,\n            'xyz_range': {\n                'min': gaussians['xyz'].min(axis=0).tolist() if num_gaussians > 0 else None,\n                'max': gaussians['xyz'].max(axis=0).tolist() if num_gaussians > 0 else None,\n                'mean': gaussians['xyz'].mean(axis=0).tolist() if num_gaussians > 0 else None\n            },\n            'opacity': {\n                'min': float(gaussians['opacity'].min()) if num_gaussians > 0 else None,\n                'max': float(gaussians['opacity'].max()) if num_gaussians > 0 else None,\n                'mean': float(gaussians['opacity'].mean()) if num_gaussians > 0 else None\n            }\n        }\n        return stats\n\n\n# 演示初始化 - Demonstrate Initialization\nprint("=" * 60)\nprint("直接3DGS初始化演示")\nprint("=" * 60)\n\n# 模拟VGGT输出 - Simulate VGGT outputs\nN, H, W = 3, 512, 640\n\n# 深度图 - Depth maps (meters)\ndepth_maps = []\nfor i in range(N):\n    # Create a plane at different depths\n    depth = np.ones((H, W)) * (2.0 + i * 0.5)\n    depth += np.random.randn(H, W) * 0.05  # Add noise\n    depth_maps.append(depth)\n\n# RGB图像 - RGB images\ncolors = np.random.rand(N, 3, H, W).astype(np.float32)\n\n# 置信度 - Confidence maps\nconfidences = []\nfor i in range(N):\n    # Higher confidence in center\n    y, x = np.ogrid[:H, :W]\n    center_dist = np.sqrt((x - W/2)**2 + (y - H/2)**2)\n    conf = np.exp(-center_dist / (min(H, W) / 3))\n    confidences.append(conf)\n\n# 相机参数 - Camera parameters\nintrinsics = []\nextrinsics = []\nfor i in range(N):\n    # Intrinsics\n    K = np.array([[500, 0, W/2],\n                  [0, 500, H/2],\n                  [0, 0, 1]], dtype=np.float32)\n    intrinsics.append(K)\n    \n    # Extrinsics: camera looking at origin from different angles\n    angle = i * 2 * np.pi / N\n    cam_pos = np.array([3*np.cos(angle), 0.5, 3*np.sin(angle)])\n    forward = -cam_pos / np.linalg.norm(cam_pos)\n    right = np.cross(forward, np.array([0, 1, 0]))\n    up = np.cross(right, forward)\n    R = np.stack([right, up, -forward], axis=0)\n    E = np.eye(4)\n    E[:3, :3] = R\n    E[:3, 3] = -R @ cam_pos\n    extrinsics.append(E)\n\n# 初始化高斯 - Initialize Gaussians\ninitializer = GaussianInitializer(subsample=16, confidence_threshold=0.3)\ngaussians = initializer.initialize_from_vggt(\n    depth_maps, colors, confidences, intrinsics, extrinsics, (H, W)\n)\n\n# 打印统计 - Print statistics\nstats = initializer.get_statistics(gaussians)\nprint(f"\n初始化统计:")\nprint(f"  高斯数量: {stats['num_gaussians']:,}")\nprint(f"  XYZ范围:")\nprint(f"    Min: [{stats['xyz_range']['min'][0]:.2f}, {stats['xyz_range']['min'][1]:.2f}, {stats['xyz_range']['min'][2]:.2f}]")\nprint(f"    Max: [{stats['xyz_range']['max'][0]:.2f}, {stats['xyz_range']['max'][1]:.2f}, {stats['xyz_range']['max'][2]:.2f}]")\nprint(f"    Mean: [{stats['xyz_range']['mean'][0]:.2f}, {stats['xyz_range']['mean'][1]:.2f}, {stats['xyz_range']['mean'][2]:.2f}]")\nprint(f"  Opacity范围: [{stats['opacity']['min']:.2f}, {stats['opacity']['max']:.2f}]")\nprint(f"  平均Opacity: {stats['opacity']['mean']:.2f}")\n\n# 可视化 - Visualize\nfig, axes = plt.subplots(2, 3, figsize=(15, 10))\n\n# 绘制点云 - Plot point cloud\nax = axes[0, 0]\nax.scatter(gaussians['xyz'][:, 0], gaussians['xyz'][:, 2], c=gaussians['rgb'], s=1, alpha=0.5)\nax.set_xlabel('X')\nax.set_ylabel('Z')\nax.set_title('Top View (X-Z)')\nax.set_aspect('equal')\nax.grid(True, alpha=0.3)\n\nax = axes[0, 1]\nax.scatter(gaussians['xyz'][:, 0], gaussians['xyz'][:, 1], c=gaussians['rgb'], s=1, alpha=0.5)\nax.set_xlabel('X')\nax.set_ylabel('Y')\nax.set_title('Front View (X-Y)')\nax.set_aspect('equal')\nax.grid(True, alpha=0.3)\n\nax = axes[0, 2]\nax.scatter(gaussians['xyz'][:, 2], gaussians['xyz'][:, 1], c=gaussians['rgb'], s=1, alpha=0.5)\nax.set_xlabel('Z')\nax.set_ylabel('Y')\nax.set_title('Side View (Z-Y)')\nax.set_aspect('equal')\nax.grid(True, alpha=0.3)\n\n# 参数分布 - Parameter distributions\nax = axes[1, 0]\nax.hist(gaussians['opacity'], bins=50, alpha=0.7, color='blue', edgecolor='black')\nax.set_xlabel('Opacity (before sigmoid)')\nax.set_ylabel('Count')\nax.set_title('Opacity Distribution')\nax.grid(True, alpha=0.3)\n\nax = axes[1, 1]\nax.hist(gaussians['scaling'].flatten(), bins=50, alpha=0.7, color='green', edgecolor='black')\nax.set_xlabel('Scaling')\nax.set_ylabel('Count')\nax.set_title('Scaling Distribution')\nax.grid(True, alpha=0.3)\n\nax = axes[1, 2]\nax.hist(np.linalg.norm(gaussians['rotation'] - [1,0,0,0], axis=1), bins=50, alpha=0.7, color='red', edgecolor='black')\nax.set_xlabel('Distance from Identity')\nax.set_ylabel('Count')\nax.set_title('Rotation Distribution (mostly identity)')\nax.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.savefig('gaussian_initialization.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("\n直接初始化完成!")\nprint("你可以将这些参数传递给gsplat或其他3DGS实现")

## 4. Depth-based Point Cloud Generation\n\nThe core operation for 3D reconstruction from depth is **unprojection**: converting 2D pixels with depth to 3D points.\n\n### Mathematical Formulation\n\nGiven:\n- Pixel coordinates $(u, v)$\n- Depth $d$ (distance along optical axis)\n- Intrinsics $K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$\n\n**Step 1: Convert to normalized camera coordinates**\n\n$$\nX_c = \frac{(u - c_x) \cdot d}{f_x}, \quad\nY_c = \frac{(v - c_y) \cdot d}{f_y}, \quad\nZ_c = d\n$$\n\n**Step 2: Transform to world coordinates**\n\n$$\nP_w = R^T P_c - R^T t\n$$\n\nwhere $E = [R | t]$ is the camera extrinsic matrix (world-to-camera).\n\n### Implementation Considerations\n\n1. **Depth scale**: VGGT outputs relative depth, need to convert to metric\n2. **Confidence weighting**: Filter points with low confidence\n3. **Depth boundaries**: Clamp extreme depth values\n4. **Edge handling**: Careful at image boundaries

In [5]:
# 深度反投影可视化 - Depth Unprojection Visualization\n\ndef unproject_depth_vectorized(depth_map, K, subsample=4):\n    """\n    向量化反投影实现\n    Vectorized depth unprojection\n    """\n    H, W = depth_map.shape\n    \n    # 创建像素网格\n    u = np.arange(0, W, subsample)\n    v = np.arange(0, H, subsample)\n    uu, vv = np.meshgrid(u, v)\n    \n    # 采样深度\n    d = depth_map[vv, uu]\n    \n    # 反投影公式\n    fx, fy = K[0, 0], K[1, 1]\n    cx, cy = K[0, 2], K[1, 2]\n    \n    X = (uu - cx) * d / fx\n    Y = (vv - cy) * d / fy\n    Z = d\n    \n    # 堆叠为3D点\n    points = np.stack([X, Y, Z], axis=-1)  # [H', W', 3]\n    \n    return points, uu, vv\n\n\n# 创建示例深度图\nH, W = 240, 320\n\n# 场景: 一个平面和一个球体\n# Scene: A plane and a sphere\ndepth_map = np.ones((H, W)) * 5.0  # Background plane at 5m\n\n# 添加球体 - Add sphere\ncenter_y, center_x = H // 2, W // 2\nradius = 60\ny, x = np.ogrid[:H, :W]\ndist_from_center = np.sqrt((x - center_x)**2 + (y - center_y)**2)\nsphere_mask = dist_from_center < radius\n\n# 球体深度\nfor i in range(H):\n    for j in range(W):\n        if sphere_mask[i, j]:\n            # Sphere surface\n            dy = (i - center_y) / radius\n            dx = (j - center_x) / radius\n            if dx**2 + dy**2 < 1:\n                dz = np.sqrt(1 - dx**2 - dy**2)\n                depth_map[i, j] = 3.0 - dz * 0.5  # Sphere at 3m\n\n# 添加噪声\ndepth_map += np.random.randn(H, W) * 0.02\n\n# 相机内参\nK = np.array([[300, 0, W/2],\n              [0, 300, H/2],\n              [0, 0, 1]])\n\n# 反投影\npoints, uu, vv = unproject_depth_vectorized(depth_map, K, subsample=2)\n\n# 可视化\nfig = plt.figure(figsize=(16, 5))\n\n# 1. 原始深度图\nax1 = fig.add_subplot(131)\nim = ax1.imshow(depth_map, cmap='viridis')\nax1.set_title('Depth Map', fontsize=12, fontweight='bold')\nax1.set_xlabel('Pixel X')\nax1.set_ylabel('Pixel Y')\nplt.colorbar(im, ax=ax1, label='Depth (m)')\n\n# 标记采样点\nax1.scatter(uu[::10, ::10], vv[::10, ::10], c='red', s=1, alpha=0.5)\n\n# 2. 3D点云 - 俯视图\nax2 = fig.add_subplot(132, projection='3d')\nX, Y, Z = points[..., 0], points[..., 1], points[..., 2]\ncolors = plt.cm.viridis((Z - Z.min()) / (Z.max() - Z.min()))\nax2.scatter(X.flatten(), Y.flatten(), Z.flatten(), c=colors.reshape(-1, 4), s=1, alpha=0.6)\nax2.set_xlabel('X (m)')\nax2.set_ylabel('Y (m)')\nax2.set_zlabel('Z (m)')\nax2.set_title('3D Point Cloud\n(Camera Coordinates)', fontsize=12, fontweight='bold')\nax2.view_init(elev=20, azim=-60)\n\n# 3. 反投影过程可视化\nax3 = fig.add_subplot(133)\n\n# 绘制相机和光线\nax3.set_xlim(-2, 2)\nax3.set_ylim(0, 6)\nax3.set_aspect('equal')\nax3.set_xlabel('X (m)')\nax3.set_ylabel('Z (m)')\nax3.set_title('Unprojection Geometry\n(Side View)', fontsize=12, fontweight='bold')\n\n# 相机位置\nax3.scatter([0], [0], c='red', s=200, marker='o', zorder=5)\nax3.text(0.1, -0.3, 'Camera', fontsize=10, fontweight='bold')\n\n# 图像平面\nimg_plane_z = 1.0\nax3.axhline(y=img_plane_z, color='blue', linestyle='--', alpha=0.5, label='Image Plane')\n\n# 绘制几条示例光线\nsample_pixels = [(W//2, H//4), (W//3, H//2), (2*W//3, H//2)]\nfor u_samp, v_samp in sample_pixels:\n    d = depth_map[v_samp, u_samp]\n    X_pt = (u_samp - K[0, 2]) * d / K[0, 0]\n    Z_pt = d\n    \n    # 绘制光线\n    ax3.plot([0, X_pt], [0, Z_pt], 'g-', alpha=0.5, linewidth=1)\n    \n    # 绘制点\n    ax3.scatter([X_pt], [Z_pt], c='green', s=50, zorder=4)\n\n# 绘制图像平面上的点\nfor u_samp, v_samp in sample_pixels:\n    X_img = (u_samp - K[0, 2]) * img_plane_z / K[0, 0]\n    ax3.scatter([X_img], [img_plane_z], c='blue', s=50, marker='s', zorder=4)\n\nax3.legend(loc='upper right')\nax3.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.savefig('depth_unprojection.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("\n深度反投影演示")\nprint(f"  深度图尺寸: {depth_map.shape}")\nprint(f"  采样间隔: 2 pixels")\nprint(f"  生成的3D点: {points.size // 3:,}")\nprint(f"  XYZ范围: X=[{X.min():.2f}, {X.max():.2f}], Y=[{Y.min():.2f}, {Y.max():.2f}], Z=[{Z.min():.2f}, {Z.max():.2f}]")

## 5. Camera Parameter Extraction\n\nVGGT outputs camera parameters in a specific format that needs conversion for use in 3DGS.\n\n### VGGT Camera Output Format\n\nVGGT outputs `pose_encoding` with shape `[B, S, 9]`:\n- **absT** [3]: Absolute translation in world coordinates\n- **quaR** [4]: Rotation as quaternion (w, x, y, z)\n- **FoV** [2]: Field of view (horizontal, vertical) in degrees\n\n### Conversion Pipeline\n\n```\nVGGT Output → Camera Intrinsics + Extrinsics\n│\n├─ FoV → focal length: f = (W/2) / tan(FoV_h/2)\n├─ quaR → Rotation matrix R\n└─ absT → Translation t\n│\n└─ (R, t) → 4×4 extrinsic matrix\n```\n\n### Intrinsics from FoV\n\n$$\nf_x = \frac{W}{2 \tan(\text{FoV}_h / 2)}, \quad\nf_y = \frac{H}{2 \tan(\text{FoV}_v / 2)}\n$$\n\n$$\nc_x = W / 2, \quad c_y = H / 2\n$$\n\n### Extrinsics from quaR and absT\n\n$$\nE = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix}\n$$\n\nwhere $R$ is the rotation matrix from quaternion $q = (w, x, y, z)$.

In [6]:
# 相机参数提取 - Camera Parameter Extraction\n\nclass CameraParameterExtractor:\n    """\n    从VGGT输出提取相机参数\n    Extract camera parameters from VGGT outputs\n    """\n    \n    @staticmethod\n    def pose_encoding_to_intrinsics_extrinsics(pose_encoding, image_size):\n        """\n        从VGGT pose encoding提取内参和外参\n        Extract intrinsics and extrinsics from VGGT pose encoding\n        \n        Args:\n            pose_encoding: [N, 9] or [9] - (absT[3], quaR[4], FoV[2])\n            image_size: (W, H) tuple\n        \n        Returns:\n            intrinsics: [N, 3, 3] camera intrinsics\n            extrinsics: [N, 4, 4] camera extrinsics\n        """\n        if pose_encoding.ndim == 1:\n            pose_encoding = pose_encoding[None, :]\n        \n        N = len(pose_encoding)\n        W, H = image_size\n        \n        # 提取组件\n        absT = pose_encoding[:, :3]  # [N, 3]\n        quaR = pose_encoding[:, 3:7]  # [N, 4]\n        FoV = pose_encoding[:, 7:9]  # [N, 2] in degrees\n        \n        # 计算内参\n        FoV_rad = np.deg2rad(FoV)\n        fx = (W / 2) / np.tan(FoV_rad[:, 0] / 2)\n        fy = (H / 2) / np.tan(FoV_rad[:, 1] / 2)\n        cx = np.full(N, W / 2)\n        cy = np.full(N, H / 2)\n        \n        intrinsics = np.zeros((N, 3, 3))\n        intrinsics[:, 0, 0] = fx\n        intrinsics[:, 1, 1] = fy\n        intrinsics[:, 0, 2] = cx\n        intrinsics[:, 1, 2] = cy\n        intrinsics[:, 2, 2] = 1\n        \n        # 计算外参\n        extrinsics = np.zeros((N, 4, 4))\n        for i in range(N):\n            R = CameraParameterExtractor.quaternion_to_rotation_matrix(quaR[i])\n            t = absT[i]\n            extrinsics[i, :3, :3] = R\n            extrinsics[i, :3, 3] = t\n            extrinsics[i, 3, 3] = 1\n        \n        return intrinsics, extrinsics\n    \n    @staticmethod\n    def quaternion_to_rotation_matrix(q):\n        """\n        四元数转换为旋转矩阵\n        Convert quaternion to rotation matrix\n        """\n        q = q / np.linalg.norm(q)\n        w, x, y, z = q\n        \n        R = np.array([[1 - 2*(y**2 + z**2), 2*(x*y - w*z), 2*(x*z + w*y)],\n                      [2*(x*y + w*z), 1 - 2*(x**2 + z**2), 2*(y*z - w*x)],\n                      [2*(x*z - w*y), 2*(y*z + w*x), 1 - 2*(x**2 + y**2)]])\n        \n        return R\n    \n    @staticmethod\n    def get_camera_frustum(extrinsic, intrinsic, image_size, scale=1.0):\n        """\n        获取相机视锥体顶点\n        Get camera frustum vertices for visualization\n        """\n        W, H = image_size\n        fx, fy = intrinsic[0, 0], intrinsic[1, 1]\n        cx, cy = intrinsic[0, 2], intrinsic[1, 2]\n        \n        # 图像平面四个角\n        # Image corners in camera space\n        d = scale  # Distance for visualization\n        corners = np.array([\n            [(0 - cx) * d / fx, (0 - cy) * d / fy, d, 1],\n            [(W - cx) * d / fx, (0 - cy) * d / fy, d, 1],\n            [(W - cx) * d / fx, (H - cy) * d / fy, d, 1],\n            [(0 - cx) * d / fx, (H - cy) * d / fy, d, 1],\n            [0, 0, 0, 1]  # Camera center\n        ]).T  # [4, 5]\n        \n        # 转换到世界坐标\n        extrinsic_inv = np.linalg.inv(extrinsic)\n        corners_world = (extrinsic_inv @ corners).T  # [5, 4]\n        \n        return corners_world[:, :3]\n\n\n# 演示相机参数提取\nprint("=" * 60)\nprint("相机参数提取演示")\nprint("=" * 60)\n\n# 模拟VGGT输出\nnum_cameras = 4\nimage_size = (1920, 1080)\n\n# 创建模拟pose encoding\npose_encodings = []\nfor i in range(num_cameras):\n    angle = i * 2 * np.pi / num_cameras\n    \n    # Translation: camera in a circle\n    absT = np.array([3*np.cos(angle), 1.0, 3*np.sin(angle)])\n    \n    # Rotation: looking at origin\n    forward = -absT / np.linalg.norm(absT)\n    right = np.cross(forward, np.array([0, 1, 0]))\n    up = np.cross(right, forward)\n    R = np.stack([right, up, -forward], axis=0)\n    \n    # Convert R to quaternion\n    trace = np.trace(R)\n    if trace > 0:\n        s = 0.5 / np.sqrt(trace + 1.0)\n        qw = 0.25 / s\n        qx = (R[2, 1] - R[1, 2]) * s\n        qy = (R[0, 2] - R[2, 0]) * s\n        qz = (R[1, 0] - R[0, 1]) * s\n    else:\n        qw, qx, qy, qz = 1, 0, 0, 0\n    quaR = np.array([qw, qx, qy, qz])\n    \n    # FoV: typical camera\n    FoV = np.array([60, 45])  # horizontal, vertical in degrees\n    \n    pose_encodings.append(np.concatenate([absT, quaR, FoV]))\n\npose_encodings = np.array(pose_encodings)\n\n# 提取参数\nextractor = CameraParameterExtractor()\nintrinsics, extrinsics = extractor.pose_encoding_to_intrinsics_extrinsics(\n    pose_encodings, image_size\n)\n\nprint(f"\n输入: {num_cameras} cameras")\nprint(f"Pose encoding shape: {pose_encodings.shape}")\nprint(f"\n提取的内参 shape: {intrinsics.shape}")\nprint(f"提取的外参 shape: {extrinsics.shape}")\n\nprint("\n第一个相机的参数:")\nprint(f"  内参 K:\n{intrinsics[0]}")\nprint(f"  外参 E:\n{extrinsics[0]}")\nprint(f"  fx={intrinsics[0,0,0]:.1f}, fy={intrinsics[0,1,1]:.1f}")\nprint(f"  cx={intrinsics[0,0,2]:.1f}, cy={intrinsics[0,1,2]:.1f}")\n\n# 可视化相机位置\nfig, axes = plt.subplots(1, 2, figsize=(16, 7))\n\n# 3D视图\nax = fig.add_subplot(121, projection='3d')\n\n# 绘制相机位置\ncamera_centers = []\nfor i in range(num_cameras):\n    E_inv = np.linalg.inv(extrinsics[i])\n    center = E_inv[:3, 3]\n    camera_centers.append(center)\n    \n    # 绘制相机位置\n    ax.scatter(*center, c=f'C{i}', s=200, marker='o', label=f'Camera {i+1}')\n    \n    # 绘制视锥体\n    frustum = extractor.get_camera_frustum(\n        extrinsics[i], intrinsics[i], image_size, scale=0.5\n    )\n    \n    # 绘制视锥体边\n    edges = [[0, 4], [1, 4], [2, 4], [3, 4], [0, 1], [1, 2], [2, 3], [3, 0]]\n    for edge in edges:\n        pts = frustum[edge]\n        ax.plot3D(pts[:, 0], pts[:, 1], pts[:, 2], f'C{i}', alpha=0.5)\n    \n    # 绘制图像平面\n    img_plane = frustum[:4]\n    ax.plot3D(\n        list(img_plane[[0,1,2,3,0], 0]),\n        list(img_plane[[0,1,2,3,0], 1]),\n        list(img_plane[[0,1,2,3,0], 2]),\n        f'C{i}', alpha=0.3\n    )\n\ncamera_centers = np.array(camera_centers)\nax.scatter([0], [0], [0], c='red', s=100, marker='*', label='Origin')\nax.set_xlabel('X')\nax.set_ylabel('Y')\nax.set_zlabel('Z')\nax.set_title('Camera Frustums', fontsize=12, fontweight='bold')\nax.legend()\n\n# 俯视图\nax2 = axes[1]\nax2.scatter(camera_centers[:, 0], camera_centers[:, 2], c='blue', s=200, marker='o')\nfor i, center in enumerate(camera_centers):\n    ax2.annotate(f'Cam {i+1}', (center[0], center[2]), xytext=(5, 5),\n                textcoords='offset points', fontsize=9)\nax2.scatter([0], [0], c='red', s=200, marker='*', label='Origin')\nax2.set_xlabel('X (m)')\nax2.set_ylabel('Z (m)')\nax2.set_title('Top View (X-Z)', fontsize=12, fontweight='bold')\nax2.set_aspect('equal')\nax2.grid(True, alpha=0.3)\nax2.legend()\n\nplt.tight_layout()\nplt.savefig('camera_parameters.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("\n相机参数提取完成!")

## 6. Confidence Filtering for Point Clouds\n\nNot all depth predictions are equally reliable. Confidence filtering helps create cleaner point clouds.\n\n### Filtering Strategies\n\n1. **Threshold-based filtering**: Keep points with confidence > threshold\n2. **Quantile filtering**: Keep top K% most confident points\n3. **Spatial filtering**: Remove isolated points\n4. **Depth consistency**: Check depth consistency across views\n\n### Confidence Sources\n\nVGGT provides confidence maps from:\n- **Depth head**: Confidence per pixel\n- **Point head**: Confidence for 3D points\n- **Track head**: Point tracking confidence\n\n### Quality Metrics\n\nAfter filtering, we can compute:\n- **Point count**: Number of remaining points\n- **Density**: Points per unit volume\n- **Coverage**: Percentage of scene covered\n- **Average confidence**: Quality indicator

In [7]:
# 置信度过滤实现 - Confidence Filtering Implementation\n\nclass ConfidenceFilter:\n    """\n    基于置信度的点云过滤\n    Point cloud filtering based on confidence\n    """\n    \n    def __init__(self, strategy='threshold', threshold=0.5, quantile=None):\n        """\n        Args:\n            strategy: 'threshold', 'quantile', or 'adaptive'\n            threshold: confidence threshold\n            quantile: if strategy='quantile', keep top quantile fraction\n        """\n        self.strategy = strategy\n        self.threshold = threshold\n        self.quantile = quantile\n    \n    def filter_points(self, points, confidences, colors=None):\n        """\n        根据置信度过滤点\n        Filter points based on confidence\n        """\n        if self.strategy == 'threshold':\n            mask = confidences >= self.threshold\n        elif self.strategy == 'quantile':\n            q = np.quantile(confidences, 1 - self.quantile)\n            mask = confidences >= q\n        elif self.strategy == 'adaptive':\n            # Use Otsu's method threshold\n            from skimage.filters import threshold_otsu\n            try:\n                thresh = threshold_otsu(confidences)\n                mask = confidences >= thresh\n            except:\n                mask = confidences >= self.threshold\n        else:\n            raise ValueError(f"Unknown strategy: {self.strategy}")\n        \n        filtered_points = points[mask]\n        filtered_confidences = confidences[mask]\n        \n        if colors is not None:\n            filtered_colors = colors[mask]\n            return filtered_points, filtered_confidences, filtered_colors, mask\n        \n        return filtered_points, filtered_confidences, mask\n    \n    def remove_outliers(self, points, colors=None, nb_neighbors=20, std_ratio=2.0):\n        """\n        使用统计滤波器移除离群点\n        Remove outliers using statistical filter\n        """\n        from sklearn.neighbors import NearestNeighbors\n        \n        # 找到每个点的邻居\n        nbrs = NearestNeighbors(n_neighbors=nb_neighbors + 1).fit(points)\n        distances, indices = nbrs.kneighbors(points)\n        \n        # 计算平均距离\n        avg_distances = distances[:, 1:].mean(axis=1)\n        \n        # 基于标准差过滤\n        mean_dist = avg_distances.mean()\n        std_dist = avg_distances.std()\n        mask = avg_distances < (mean_dist + std_ratio * std_dist)\n        \n        if colors is not None:\n            return points[mask], colors[mask], mask\n        return points[mask], mask\n    \n    def filter_by_depth_consistency(self, depth_maps, extrinsics, intrinsics,\n                                   threshold=0.1):\n        """\n        基于多视角深度一致性过滤\n        Filter based on multi-view depth consistency\n        """\n        # This is a placeholder for multi-view consistency check\n        # In practice, you'd project points from one view to another\n        # and check if depths agree\n        pass\n\n\n# 演示置信度过滤\nprint("=" * 60)\nprint("置信度过滤演示")\nprint("=" * 60)\n\n# 生成模拟点云和置信度\nnp.random.seed(42)\nnum_points = 10000\n\n# 生成中心密集的点云\ntheta = np.random.uniform(0, 2*np.pi, num_points)\nphi = np.random.uniform(0, np.pi, num_points)\nr = np.random.exponential(2, num_points)\n\npoints = np.stack([\n    r * np.sin(phi) * np.cos(theta),\n    r * np.sin(phi) * np.sin(theta),\n    r * np.cos(phi)\n], axis=1)\n\n# 生成置信度 (中心点置信度高)\ndistances = np.linalg.norm(points, axis=1)\nconfidences = np.exp(-distances / 3) + np.random.randn(num_points) * 0.1\nconfidences = np.clip(confidences, 0, 1)\n\n# 颜色\ncolors = np.random.rand(num_points, 3)\n\n# 测试不同过滤策略\nstrategies = [\n    ('threshold', 0.5),\n    ('threshold', 0.7),\n    ('quantile', 0.3),\n]\n\nresults = []\nfor strategy, param in strategies:\n    if strategy == 'threshold':\n        filt = ConfidenceFilter(strategy='threshold', threshold=param)\n        label = f'Threshold={param}'\n    else:\n        filt = ConfidenceFilter(strategy='quantile', quantile=param)\n        label = f'Top {int(param*100)}%'\n    \n    fp, fc, fcol, mask = filt.filter_points(points, confidences, colors)\n    results.append((label, fp, fc, fcol))\n    print(f"{label}: {len(fp):,} / {num_points:,} points ({len(fp)/num_points*100:.1f}%)")\n\n# 可视化\nfig, axes = plt.subplots(2, 3, figsize=(18, 12))\n\n# 原始数据\nax = axes[0, 0]\nscatter = ax.scatter(points[:, 0], points[:, 2], c=confidences, cmap='RdYlGn',\n                    s=1, alpha=0.5, vmin=0, vmax=1)\nax.set_title(f'Original\n{num_points:,} points', fontsize=11, fontweight='bold')\nax.set_xlabel('X')\nax.set_ylabel('Z')\nax.set_aspect('equal')\nplt.colorbar(scatter, ax=ax, label='Confidence')\n\n# 过滤结果\nfor idx, (label, fp, fc, fcol) in enumerate(results):\n    ax = axes[0, idx + 1]\n    ax.scatter(fp[:, 0], fp[:, 2], c=fc, cmap='RdYlGn', s=2, alpha=0.7, vmin=0, vmax=1)\n    ax.set_title(f'{label}\n{len(fp):,} points', fontsize=11, fontweight='bold')\n    ax.set_xlabel('X')\n    ax.set_ylabel('Z')\n    ax.set_aspect('equal')\n\n# 置信度分布\nax = axes[1, 0]\nax.hist(confidences, bins=50, alpha=0.7, color='blue', edgecolor='black')\nax.axvline(x=0.5, color='r', linestyle='--', label='Threshold=0.5')\nax.axvline(x=0.7, color='g', linestyle='--', label='Threshold=0.7')\nax.set_xlabel('Confidence')\nax.set_ylabel('Count')\nax.set_title('Confidence Distribution', fontsize=11, fontweight='bold')\nax.legend()\nax.grid(True, alpha=0.3)\n\n# 保留比例\nax = axes[1, 1]\nthresholds = np.linspace(0, 1, 50)\nretained = [(confidences >= t).sum() / len(confidences) * 100 for t in thresholds]\nax.plot(thresholds, retained, 'b-', linewidth=2)\nax.fill_between(thresholds, retained, alpha=0.3)\nax.set_xlabel('Confidence Threshold')\nax.set_ylabel('Retained Points (%)')\nax.set_title('Retention vs Threshold', fontsize=11, fontweight='bold')\nax.grid(True, alpha=0.3)\n\n# 3D散点图\nax = fig.add_subplot(236, projection='3d')\nlabel, fp, fc, fcol = results[1]  # Use threshold=0.7 result\nax.scatter(fp[:, 0], fp[:, 1], fp[:, 2], c=fc, cmap='RdYlGn', s=1, alpha=0.6)\nax.set_xlabel('X')\nax.set_ylabel('Y')\nax.set_zlabel('Z')\nax.set_title(f'Filtered 3D View\n({label})', fontsize=11, fontweight='bold')\n\nplt.tight_layout()\nplt.savefig('confidence_filtering.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("\n置信度过滤完成!")\nprint("建议使用自适应阈值或Top-K策略以获得最佳效果")

## 7. Complete Pipeline Implementation\n\nLet's put everything together into a complete end-to-end pipeline.\n\n### Pipeline Architecture\n\n```\nInput Images (N images)\n        ↓\n┌─────────────────────────────────┐\n│         VGGT Model              │\n│  ┌─────────────────────────┐    │\n│  │  Shared ViT Backbone    │    │\n│  └─────────────────────────┘    │\n│              ↓                  │\n│  ┌─────────────────────────┐    │\n│  │    Multi-task Heads     │    │\n│  │  • Camera Head          │    │\n│  │  • Depth Head           │    │\n│  │  • Track Head           │    │\n│  └─────────────────────────┘    │\n└─────────────────────────────────┘\n        ↓\nVGGT Outputs:\n  • Cameras: poses [N, 9]\n  • Depth maps: [N, H, W]\n  • Confidence: [N, H, W]\n  • Track (optional)\n        ↓\n┌─────────────────────────────────┐\n│    Post-processing Pipeline     │\n│  ┌─────────────────────────┐    │\n│  │  Camera Extraction      │    │\n│  │  (poses → K, E)         │    │\n│  └─────────────────────────┘    │\n│              ↓                  │\n│  ┌─────────────────────────┐    │\n│  │  Depth Unprojection     │    │\n│  │  (depth → 3D points)    │    │\n│  └─────────────────────────┘    │\n│              ↓                  │\n│  ┌─────────────────────────┐    │\n│  │  Confidence Filtering   │    │\n│  │  (remove low-conf)      │    │\n│  └─────────────────────────┘    │\n│              ↓                  │\n│  ┌─────────────────────────┐    │\n│  │  Gaussian Init          │    │\n│  │  (xyz, rgb, opacity...) │    │\n│  └─────────────────────────┘    │\n└─────────────────────────────────┘\n        ↓\nOutput: 3D Gaussians\n```

In [8]:
# 完整管道实现 - Complete Pipeline Implementation\n\nclass VGGTto3DGSPipeline:\n    """\n    完整的VGGT到3DGS管道\n    Complete pipeline from VGGT to 3DGS\n    """\n    \n    def __init__(self, confidence_threshold=0.5, subsample=8, output_mode='direct'):\n        """\n        Args:\n            confidence_threshold: minimum confidence to keep a point\n            subsample: pixel sampling stride\n            output_mode: 'direct' or 'colmap'\n        """\n        self.confidence_threshold = confidence_threshold\n        self.subsample = subsample\n        self.output_mode = output_mode\n        \n        # 组件 - Components\n        self.camera_extractor = CameraParameterExtractor()\n        self.gaussian_initializer = GaussianInitializer(\n            subsample=subsample,\n            confidence_threshold=confidence_threshold\n        )\n        self.confidence_filter = ConfidenceFilter(\n            strategy='threshold',\n            threshold=confidence_threshold\n        )\n        \n        if output_mode == 'colmap':\n            self.colmap_exporter = COLMAPExporter()\n    \n    def process(self, vggt_outputs, images, image_names=None, image_size=None):\n        """\n        处理VGGT输出\n        Process VGGT outputs\n        \n        Args:\n            vggt_outputs: dict with 'pose_encoding', 'depth_maps', 'confidences'\n            images: [N, 3, H, W] RGB images\n            image_names: list of image filenames (for COLMAP export)\n            image_size: (W, H) tuple (optional, inferred from images)\n        \n        Returns:\n            gaussians or colmap_data depending on output_mode\n        """\n        # 获取尺寸\n        if image_size is None:\n            if images.ndim == 4:\n                _, _, H, W = images.shape\n            else:\n                H, W = images.shape[-2:]\n            image_size = (W, H)\n        else:\n            W, H = image_size\n        \n        # 提取相机参数\n        print("Step 1: Extracting camera parameters...")\n        pose_encoding = vggt_outputs['pose_encoding']\n        intrinsics, extrinsics = self.camera_extractor.pose_encoding_to_intrinsics_extrinsics(\n            pose_encoding, image_size\n        )\n        print(f"  ✓ Extracted {len(intrinsics)} cameras")\n        \n        # 初始化高斯\n        print("\nStep 2: Initializing 3D Gaussians...")\n        depth_maps = vggt_outputs['depth_maps']\n        confidences = vggt_outputs['confidences']\n        \n        gaussians = self.gaussian_initializer.initialize_from_vggt(\n            depth_maps, images, confidences,\n            intrinsics, extrinsics, (H, W)\n        )\n        print(f"  ✓ Initialized {len(gaussians['xyz']):,} Gaussians")\n        \n        # 置信度过滤\n        print("\nStep 3: Applying confidence filtering...")\n        opacity_sigmoid = 1 / (1 + np.exp(-gaussians['opacity']))\n        xyz_filtered, _, rgb_filtered, mask = self.confidence_filter.filter_points(\n            gaussians['xyz'], opacity_sigmoid, gaussians['rgb']\n        )\n        \n        for key in gaussians:\n            gaussians[key] = gaussians[key][mask]\n        \n        print(f"  ✓ Retained {len(gaussians['xyz']):,} Gaussians after filtering")\n        \n        # 根据模式返回\n        if self.output_mode == 'direct':\n            print("\n✓ Pipeline complete!")\n            return gaussians\n        \n        elif self.output_mode == 'colmap':\n            print("\nStep 4: Exporting to COLMAP format...")\n            if image_names is None:\n                image_names = [f"image_{i:03d}.jpg" for i in range(len(intrinsics))]\n            \n            cameras_txt = self.colmap_exporter.export_cameras(\n                intrinsics[0], image_size  # Assuming shared intrinsics\n            )\n            images_txt = self.colmap_exporter.export_images(extrinsics, image_names)\n            points3D_txt = self.colmap_exporter.export_points3D(\n                gaussians['xyz'],\n                (gaussians['rgb'] * 255).astype(np.uint8)\n            )\n            \n            print("  ✓ COLMAP export complete!")\n            return {\n                'gaussians': gaussians,\n                'cameras_txt': cameras_txt,\n                'images_txt': images_txt,\n                'points3D_txt': points3D_txt,\n                'intrinsics': intrinsics,\n                'extrinsics': extrinsics\n            }\n    \n    def get_pipeline_summary(self, results):\n        """\n        获取管道处理摘要\n        Get pipeline processing summary\n        """\n        if self.output_mode == 'direct':\n            gaussians = results\n        else:\n            gaussians = results['gaussians']\n        \n        summary = {\n            'num_gaussians': len(gaussians['xyz']),\n            'xyz_bounds': {\n                'x': [float(gaussians['xyz'][:, 0].min()), float(gaussians['xyz'][:, 0].max())],\n                'y': [float(gaussians['xyz'][:, 1].min()), float(gaussians['xyz'][:, 1].max())],\n                'z': [float(gaussians['xyz'][:, 2].min()), float(gaussians['xyz'][:, 2].max())]\n            },\n            'opacity_stats': {\n                'mean': float(gaussians['opacity'].mean()),\n                'std': float(gaussians['opacity'].std())\n            },\n            'output_mode': self.output_mode\n        }\n        \n        return summary\n\n\n# 演示完整管道\nprint("=" * 70)\nprint("完整管道演示")\nprint("=" * 70)\n\n# 模拟VGGT输出\nN, H, W = 4, 384, 512\n\n# 模拟pose encoding\npose_encoding = []\nfor i in range(N):\n    angle = i * 2 * np.pi / N\n    absT = np.array([2*np.cos(angle), 0.5, 2*np.sin(angle)])\n    \n    forward = -absT / np.linalg.norm(absT)\n    right = np.cross(forward, np.array([0, 1, 0]))\n    up = np.cross(right, forward)\n    R = np.stack([right, up, -forward], axis=0)\n    \n    trace = np.trace(R)\n    if trace > 0:\n        s = 0.5 / np.sqrt(trace + 1.0)\n        qw = 0.25 / s\n        qx = (R[2, 1] - R[1, 2]) * s\n        qy = (R[0, 2] - R[2, 0]) * s\n        qz = (R[1, 0] - R[0, 1]) * s\n    quaR = np.array([qw, qx, qy, qz])\n    FoV = np.array([60, 45])\n    pose_encoding.append(np.concatenate([absT, quaR, FoV]))\n\n# 模拟深度和置信度\ndepth_maps = []\nconfidences = []\nimages = []\nfor i in range(N):\n    # 深度: 一个球体表面\n    depth = np.ones((H, W)) * 3.0\n    \n    # 在中心添加一些变化\n    y, x = np.ogrid[:H, :W]\n    center_dist = np.sqrt((x - W/2)**2 + (y - H/2)**2)\n    depth -= np.exp(-center_dist**2 / (2 * 50**2)) * 0.5\n    depth += np.random.randn(H, W) * 0.01\n    depth_maps.append(depth)\n    \n    # 置信度\n    conf = np.exp(-center_dist / 100)\n    confidences.append(conf)\n    \n    # 图像\n    img = np.random.rand(3, H, W).astype(np.float32)\n    images.append(img)\n\nvggt_outputs = {\n    'pose_encoding': np.array(pose_encoding),\n    'depth_maps': depth_maps,\n    'confidences': confidences\n}\nimages = np.array(images)\n\n# 运行管道 (Direct模式)\nprint("\n--- Direct Initialization Mode ---\n")\npipeline_direct = VGGTto3DGSPipeline(\n    confidence_threshold=0.4,\n    subsample=8,\n    output_mode='direct'\n)\n\ngaussians = pipeline_direct.process(vggt_outputs, images, image_size=(W, H))\n\nsummary = pipeline_direct.get_pipeline_summary(gaussians)\nprint(f"\n处理摘要:")\nprint(f"  高斯数量: {summary['num_gaussians']:,}")\nprint(f"  XYZ范围:")\nfor axis, bounds in summary['xyz_bounds'].items():\n    print(f"    {axis}: [{bounds[0]:.2f}, {bounds[1]:.2f}]")\nprint(f"  Opacity均值: {summary['opacity_stats']['mean']:.3f}")\n\n# 运行管道 (COLMAP模式)\nprint("\n\n--- COLMAP Export Mode ---\n")\npipeline_colmap = VGGTto3DGSPipeline(\n    confidence_threshold=0.4,\n    subsample=8,\n    output_mode='colmap'\n)\n\nimage_names = [f"frame_{i:03d}.jpg" for i in range(N)]\ncolmap_data = pipeline_colmap.process(\n    vggt_outputs, images, image_names=image_names, image_size=(W, H)\n)\n\nprint(f"\nCOLMAP输出文件大小:")\nprint(f"  cameras.txt: {len(colmap_data['cameras_txt'])} chars")\nprint(f"  images.txt: {len(colmap_data['images_txt'])} chars")\nprint(f"  points3D.txt: {len(colmap_data['points3D_txt'])} chars")\n\n# 可视化\nfig, axes = plt.subplots(2, 3, figsize=(18, 12))\n\n# 原始深度图\nax = axes[0, 0]\nax.imshow(depth_maps[0], cmap='viridis')\nax.set_title('Sample Depth Map', fontsize=11, fontweight='bold')\nax.axis('off')\n\n# 置信度图\nax = axes[0, 1]\nax.imshow(confidences[0], cmap='RdYlGn')\nax.set_title('Sample Confidence Map', fontsize=11, fontweight='bold')\nax.axis('off')\n\n# RGB图\nax = axes[0, 2]\nax.imshow(images[0].transpose(1, 2, 0))\nax.set_title('Sample RGB Image', fontsize=11, fontweight='bold')\nax.axis('off')\n\n# 3D点云 - 俯视图\nax = axes[1, 0]\nax.scatter(gaussians['xyz'][:, 0], gaussians['xyz'][:, 2],\n          c=gaussians['rgb'], s=1, alpha=0.5)\nax.set_xlabel('X')\nax.set_ylabel('Z')\nax.set_title('Top View (X-Z)', fontsize=11, fontweight='bold')\nax.set_aspect('equal')\nax.grid(True, alpha=0.3)\n\n# 3D点云 - 前视图\nax = axes[1, 1]\nax.scatter(gaussians['xyz'][:, 0], gaussians['xyz'][:, 1],\n          c=gaussians['rgb'], s=1, alpha=0.5)\nax.set_xlabel('X')\nax.set_ylabel('Y')\nax.set_title('Front View (X-Y)', fontsize=11, fontweight='bold')\nax.set_aspect('equal')\nax.grid(True, alpha=0.3)\n\n# 3D视图\nax = fig.add_subplot(236, projection='3d')\nax.scatter(gaussians['xyz'][:, 0], gaussians['xyz'][:, 1], gaussians['xyz'][:, 2],\n          c=gaussians['rgb'], s=1, alpha=0.5)\nax.set_xlabel('X')\nax.set_ylabel('Y')\nax.set_zlabel('Z')\nax.set_title(f'3D View\n({summary["num_gaussians"]:,} Gaussians)', fontsize=11, fontweight='bold')\n\nplt.tight_layout()\nplt.savefig('complete_pipeline.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("\n完整管道演示完成!")\nprint("你可以使用这个管道类来处理真实的VGGT输出")

## 8. Integration with gsplat\n\ngsplat is a differentiable Gaussian rasterizer. Let's see how to integrate our VGGT pipeline with it.\n\n### gsplat Gaussian Representation\n\ngsplat expects Gaussians as tensors:\n- `means`: [N, 3] - 3D positions\n- `scales`: [N, 3] - log of scales\n- `rotations`: [N, 4] - normalized quaternions (w, x, y, z)\n- `opacities`: [N, 1] - logit of opacity\n- `colors`: [N, C] - RGB or SH coefficients\n\n### Integration Code\n\nThe key is converting our numpy arrays to torch tensors with proper parameterization.

In [9]:
# gsplat集成 - gsplat Integration\n\nclass GsplatAdapter:\n    """\n    将高斯参数转换为gsplat格式\n    Convert Gaussian parameters to gsplat format\n    """\n    \n    def __init__(self, device='cpu'):\n        self.device = device\n    \n    def to_gsplat(self, gaussians, spherical_harmonics_degree=0):\n        """\n        转换到gsplat格式\n        Convert to gsplat format\n        \n        Args:\n            gaussians: dict with 'xyz', 'rgb', 'opacity', 'scaling', 'rotation'\n            spherical_harmonics_degree: 0 for RGB, >0 for SH\n        \n        Returns:\n            gsplat_gaussians: dict with gsplat-formatted tensors\n        """\n        if TORCH_AVAILABLE:\n            import torch\n        else:\n            print("Warning: PyTorch not available, returning numpy arrays")\n            return gaussians\n        \n        # 转换为torch张量\n        means = torch.from_numpy(gaussians['xyz']).float().to(self.device)\n        \n        # gsplat expects log scales\n        scales = torch.from_numpy(np.log(gaussians['scaling'])).float().to(self.device)\n        \n        # 归一化四元数\n        rotations = gaussians['rotation']\n        rotations_norm = rotations / (np.linalg.norm(rotations, axis=1, keepdims=True) + 1e-8)\n        rotations = torch.from_numpy(rotations_norm).float().to(self.device)\n        \n        # gsplat期望logit opacity\n        opacities = torch.from_numpy(gaussians['opacity']).float().to(self.device)\n        opacities = opacities.unsqueeze(-1)  # [N, 1]\n        \n        # 颜色\n        if spherical_harmonics_degree == 0:\n            # RGB直接\n            colors = torch.from_numpy(gaussians['rgb']).float().to(self.device)\n        else:\n            # 需要拟合SH系数 (这里简单用RGB作为DC系数)\n            num_coeffs = (spherical_harmonics_degree + 1) ** 2\n            colors = torch.zeros(len(means), num_coeffs, 3).to(self.device)\n            colors[:, 0] = torch.from_numpy(gaussians['rgb']).float().to(self.device)\n        \n        return {\n            'means': means,\n            'scales': scales,\n            'rotations': rotations,\n            'opacities': opacities,\n            'colors': colors,\n            'active_sh_degree': spherical_harmonics_degree\n        }\n    \n    def from_gsplat(self, gsplat_gaussians):\n        """\n        从gsplat格式转换回来\n        Convert from gsplat format\n        """\n        gaussians = {\n            'xyz': gsplat_gaussians['means'].cpu().numpy(),\n            'scaling': np.exp(gsplat_gaussians['scales'].cpu().numpy()),\n            'rotation': gsplat_gaussians['rotations'].cpu().numpy(),\n            'opacity': gsplat_gaussians['opacities'].cpu().numpy().squeeze(),\n            'rgb': gsplat_gaussians['colors'][:, 0].cpu().numpy() if gsplat_gaussians['colors'].dim() == 3\n                  else gsplat_gaussians['colors'].cpu().numpy()\n        }\n        return gaussians\n    \n    def compute_bounds(self, gsplat_gaussians):\n        """\n        计算场景边界\n        Compute scene bounds\n        """\n        means = gsplat_gaussians['means']\n        scales = torch.exp(gsplat_gaussians['scales'])\n        \n        # 考虑尺度后的边界\n        min_bounds = (means - 3 * scales).min(dim=0)[0]\n        max_bounds = (means + 3 * scales).max(dim=0)[0]\n        \n        return {\n            'min': min_bounds.cpu().numpy(),\n            'max': max_bounds.cpu().numpy(),\n            'center': ((min_bounds + max_bounds) / 2).cpu().numpy(),\n            'extent': ((max_bounds - min_bounds)).cpu().numpy()\n        }\n\n\n# 使用示例 (伪代码)\nprint("=" * 60)\nprint("gsplat集成示例")\nprint("=" * 60)\n\nif TORCH_AVAILABLE:\n    print("\n转换到gsplat格式:")\n    adapter = GsplatAdapter(device='cpu')\n    gsplat_gaussians = adapter.to_gsplat(gaussians, spherical_harmonics_degree=0)\n    \n    print(f"  means shape: {gsplat_gaussians['means'].shape}")\n    print(f"  scales shape: {gsplat_gaussians['scales'].shape}")\n    print(f"  rotations shape: {gsplat_gaussians['rotations'].shape}")\n    print(f"  opacities shape: {gsplat_gaussians['opacities'].shape}")\n    print(f"  colors shape: {gsplat_gaussians['colors'].shape}")\n    \n    bounds = adapter.compute_bounds(gsplat_gaussians)\n    print(f"\n场景边界:")\n    print(f"  Center: [{bounds['center'][0]:.2f}, {bounds['center'][1]:.2f}, {bounds['center'][2]:.2f}]")\n    print(f"  Extent: [{bounds['extent'][0]:.2f}, {bounds['extent'][1]:.2f}, {bounds['extent'][2]:.2f}]")\n    \n    print("\n\n使用gsplat进行渲染 (伪代码):")\n    print("```")\n    print("from gsplat import rasterization\n")\n    print("# 准备相机\n")\n    print("K = torch.tensor([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])\n")\n    print("E = torch.tensor(extrinsic)  # world-to-cam\n")\n    print("\n")\n    print("# 光栅化\n")\n    print("rendered = rasterization(\n")\n    print("    means=gsplat_gaussians['means'],\n")\n    print("    quats=gsplat_gaussians['rotations'],\n")\n    print("    scales=gsplat_gaussians['scales'],\n")\n    print("    opacities=torch.sigmoid(gsplat_gaussians['opacities']),\n")\n    print("    colors=gsplat_gaussians['colors'],\n")\n    print("    viewmats=E.inverse()[None],  # cam-to-world\n")\n    print("    Ks=K[None],\n")\n    print("    width=W,\n")\n    print("    height=H\n")\n    print(")\n")\n    print("```")\nelse:\n    print("\nPyTorch not available. Integration code:")\n    print("```python")\n    print("adapter = GsplatAdapter(device='cuda')")\n    print("gsplat_gaussians = adapter.to_gsplat(gaussians)")\n    print("# Now use with gsplat for rendering")\n    print("```")\n\nprint("\n集成完成!")

## Summary\n\n```\n╔═══════════════════════════════════════════════════════════════════════╗\n║              Key Takeaways - VGGT to 3DGS Integration                 ║\n╠═══════════════════════════════════════════════════════════════════════╣\n║                                                                       ║\n║  1. Two main integration approaches:                                  ║\n║     • COLMAP Export: Compatible with existing pipelines               ║\n║     • Direct Initialization: More efficient, full control             ║\n║                                                                       ║\n║  2. COLMAP export requires three files:                               ║\n║     • cameras.txt: Intrinsics (fx, fy, cx, cy)                        ║\n║     • images.txt: Extrinsics (quaternion + translation)               ║\n║     • points3D.txt: 3D points with color                              ║\n║                                                                       ║\n║  3. Direct initialization maps VGGT outputs to Gaussian params:       ║\n║     • Depth + Intrinsics → xyz positions                              ║\n║     • RGB images → colors                                             ║\n║     • Confidence → opacity                                            ║\n║     • Depth variance → scales                                         ║\n║     • Identity or normals → rotations                                 ║\n║                                                                       ║\n║  4. Depth unprojection formula:                                       ║\n║     X = (u - cx) * d / fx                                            ║\n║     Y = (v - cy) * d / fy                                            ║\n║     Z = d                                                            ║\n║                                                                       ║\n║  5. Camera extraction from VGGT:                                      ║\n║     • FoV → focal length: f = (W/2) / tan(FoV/2)                     ║\n║     • quaR → rotation matrix                                          ║\n║     • absT → translation                                              ║\n║                                                                       ║\n║  6. Confidence filtering strategies:                                  ║\n║     • Threshold: Keep points with conf > 0.5                          ║\n║     • Quantile: Keep top K% most confident                            ║\n║     • Adaptive: Use Otsu's method                                     ║\n║                                                                       ║\n║  7. Complete pipeline steps:                                          ║\n║     ① Extract cameras from pose_encoding                              ║\n║     ② Unproject depth maps to 3D points                               ║\n║     ③ Filter points by confidence                                     ║\n║     ④ Initialize Gaussian parameters                                  ║\n║     ⑤ (Optional) Export to COLMAP                                     ║\n║                                                                       ║\n║  8. gsplat integration:                                               ║\n║     • Convert scales to log space                                     ║\n║     • Normalize quaternions                                           ║\n║     • Opacities in logit space                                        ║\n║     • Use differentiable rasterizer for training                      ║\n║                                                                       ║\n║  9. Best practices:                                                   ║\n║     • Use confidence threshold 0.3-0.7                                ║\n║     • Subsample pixels (every 4-16 pixels) for efficiency             ║\n║     • Remove statistical outliers                                     ║\n║     • Normalize scene scale before training                           ║\n║                                                                       ║\n╚═══════════════════════════════════════════════════════════════════════╝\n```\n\n### What's Next?\n\n- Fine-tune Gaussian parameters with differentiable rendering\n- Implement adaptive density control (splitting/pruning)\n- Use tracking data for temporal consistency (video)\n- Optimize per-scene with appearance embeddings

## References\n\n1. **VGGT**: Video Gaussian Gaussian Transformer (2024)\n2. **3D Gaussian Splatting**: Kerbl et al., "3D Gaussian Splatting for Real-Time Radiance Field Rendering", SIGGRAPH 2023\n3. **gsplat**: https://github.com/nerfstudio-project/gsplat - Differentiable Gaussian rasterizer\n4. **COLMAP**: Schönberger & Frahm, "Structure-from-Motion Revisited", CVPR 2016\n5. **DUSt3R**: Wang et al., "DUSt3R: Geometric 3D Vision Made Easy", CVPR 2024